# Lab1 — PyTorch Foundations for Computer Vision

**Course**: Deep Learning for Image Analysis

**Class**: M2 IASD App  

**Professor**: Mehyar MLAWEH

---

## Objectives
By the end of this lab, you should be able to:

- Understand how **neurons and layers** are implemented in PyTorch
- Manipulate **tensors** and reason about shapes
- Use **autograd** to compute gradients
- Implement a **training loop** yourself
- Connect theory (neurons, loss, backprop) to actual code

⚠️ This notebook is **intentionally incomplete**.  
Whenever you see **`# TODO`**, you are expected to write code.

In [1]:
! git clone https://github.com/CharlyGuy/Tp-Apprentissage-profond-pour-l-analyse-d-image.git

Cloning into 'Tp-Apprentissage-profond-pour-l-analyse-d-image'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 23 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 127.53 KiB | 582.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.



**Deadline:** 🗓️ **Saturday, February 7th (23:59)**

## 🤖 A small (honest) note before you start

Let’s be real for a second.

 I know you **can use LLMs (ChatGPT, Copilot, Claude, etc.)** to help you with this lab.  
And yes, **I use them too**, so don’t worry 😄

👉 **You are allowed to use AI tools.**  
But here’s the deal:

- Don’t just **copy–paste** code you don’t understand  
- Take time to **read, question, and modify** what the model gives you  
- If you can solve a block **by yourself, without AI**, that’s excellent

Remember:

> AI can write code for you, but **only you can understand it** — and understanding is what matters for exams, projects, and real work.

Use these tools **as assistants, not as replacements for thinking**.

---

## 📚 Useful documentation (highly recommended)

You will often find answers faster (and more reliably) by checking the official documentation:

- **PyTorch main documentation**  
  https://pytorch.org/docs/stable/index.html

- **PyTorch tensors**  
  https://pytorch.org/docs/stable/tensors.html

- **Neural network modules (`torch.nn`)**  
  https://pytorch.org/docs/stable/nn.html

- **Loss functions** (`BCEWithLogitsLoss`, CrossEntropy, etc.)  
  https://pytorch.org/docs/stable/nn.html#loss-functions

- **Optimizers** (`SGD`, `Adam`, …)  
  https://pytorch.org/docs/stable/optim.html

If you learn how to **navigate the documentation**, you are already thinking like a real AI engineer 👌

---

## PART I

## 0) Colab setup — GPU check

**Instructions**
1. In Colab: `Runtime → Change runtime type to GPU T4`
2. Select **GPU**
3. Save and restart runtime

Then run the cell below.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# TODO: set the device correctly (cuda if available, else cpu)
# device = ...

# print("Using device:", device)


PyTorch version: 2.9.0+cu126
CUDA available: True


## 1) Imports and reproducibility


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# TODO: fix the random seed for reproducibility
torch.manual_seed(50)


## 2) PyTorch tensors and shapes

Tensors are multi-dimensional arrays that support:
- GPU acceleration
- automatic differentiation

Understanding **shapes** is critical in deep learning.


In [ ]:
# Examples
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.randn(4, 5)
print("a shape:", a.shape)
print("b shape:", b.shape)


a shape: torch.Size([3])
b shape: torch.Size([4, 5])


### 🔍 Question (answer inside the markdown)
- How many dimensions does tensor `b` have? :2
- What does each dimension represent conceptually?
This means:
1st dimension (4): the number of rows
2nd dimension (5): the number of columns

### ✅Tensor operations

Complete the following:

1. Create a tensor `x` of shape `(8, 3)` with random values  
2. Compute:
   - the **mean of each column**
   - the **L2 norm of each row**
3. Normalize `x` **row-wise** using the L2 norm

In [ ]:
# TODO: create x
x = torch.randn(8, 3)

# TODO: column mean
col_mean =torch.mean(x)

# TODO: row-wise L2 norm
row_norm = torch.norm(x, p=2, dim=1)


# TODO: normalized tensor
x_normalized = x / row_norm.unsqueeze(1)


print(x.shape, col_mean.shape, row_norm.shape, x_normalized.shape)


torch.Size([8, 3]) torch.Size([]) torch.Size([8]) torch.Size([8, 3])


In [ ]:
# Examples
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.randn(4, 5)

print("a shape:", a.shape)
print("b shape:", b.shape)


a shape: torch.Size([3])
b shape: torch.Size([4, 5])


## 3) Artificial neuron — from math to code

A neuron computes:

$$
z = \sum_i w_i x_i + b
$$

Then applies an activation function:

$$
y = g(z)
$$

This section connects directly to the theory seen in class.


In [ ]:
x = torch.tensor([1.0, -2.0, 3.0])
w = torch.tensor([0.2, 0.4, -0.1])
b = torch.tensor(0.1)

z = torch.sum(x * w) + b
z


tensor(-0.8000)

### Activation functions

1. Implement **ReLU**
2. Implement **Sigmoid**
3. Apply both to `z` and compare the outputs

Which activation preserves negative values?


In [ ]:
from IPython.core.display import Math
import numpy as np
# TODO
def relu(z):
  return torch.maximum(torch.tensor(0.0), z)
#     ...

def sigmoid(z):
  return 1 / (1 + torch.exp(-z))


y_relu = relu(z)
y_sigmoid =sigmoid(z)
y_relu, y_sigmoid


(tensor(0.), tensor(0.3100))

## 4) Autograd and gradients

PyTorch uses **automatic differentiation** to compute gradients
using the **chain rule** (backpropagation).


In [ ]:
x = torch.tensor([1.0, 2.0, -1.0], requires_grad=True)
w = torch.tensor([0.5, -0.3, 0.8], requires_grad=True)
b = torch.tensor(0.2, requires_grad=True)

z = torch.sum(x * w) + b
loss = (z - 1.0) ** 2

loss.backward()

print("loss:", loss.item())
print("grad w:", w.grad)
print("grad b:", b.grad)


loss: 2.890000104904175
grad w: tensor([-3.4000, -6.8000,  3.4000])
grad b: tensor(-3.4000)


### 🔍 Conceptual question

- If `b.grad > 0`, should `b` increase or decrease after a gradient descent step?: b will decrease


Explain **why** in one sentence:In gradient descent, parameters are updated by subtracting the gradient, so if b.grad > 0, subtracting a positive number makes b smaller


## 5) Toy classification dataset

We create a **linearly separable** dataset.

Label rule:
- class = 1 if `x₁ + x₂ + x₃ > 0`
- else class = 0

This mimics a very simple classification problem.


In [ ]:
# TODO: generate a dataset of size N=500 with 3 features
import torch

# paramètres
N = 500
D = 3
torch.manual_seed(50)
# données
X = torch.randn(N, D)        # shape (500, 3)
y = torch.randint(0,2,(N,1) ) # shape (500, 1)

# TODO: split into train (80%) and validation (20%)
indices = torch.randperm(N)
# taille des splits
n_train = int(0.8 * N)

train_idx = indices[:n_train]
val_idx = indices[n_train:]

# split effectif
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]


## 6) Model definition

We define a small **MLP** (fully-connected network):

`3 → 16 → 8 → 1`

Activation: ReLU  
Output: raw logits (no sigmoid)


In [ ]:
import torch
import torch.nn as nn
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
              nn.Linear(3, 16),
              nn.ReLU(),
              nn.Linear(16,8),
              nn.ReLU(),
              nn.Linear(8,1)

            # TODO: Linear 3 → 16
            # TODO: ReLU
            # TODO: Linear 16 → 8
            # TODO: ReLU
            # TODO: Linear 8 → 1
        )

    def forward(self, x):
        return self.net(x)

# TODO: create model and move it to the GPU
# model = ...
# création du modèle
model = MLP()

# choix du device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# déplacement du modèle sur le device
model = model.to(device)


###  parameters

1. Compute **by hand** the total number of parameters
2. Verify your answer using PyTorch


In [ ]:
#TODO: count parameters with PyTorch
total_params = sum(p.numel() for p in model.parameters())
total_params


209

## 7) Training loop

You must complete the full training loop:
- forward pass
- loss computation
- backward pass
- optimizer step

Loss: `BCEWithLogitsLoss`
Optimizer: `SGD`


In [ ]:
import torch.optim as optim
# TODO: move data to device
# X_train_d = ...
# y_train_d = ...
# X_val_d = ...
# y_val_d = ...
y_train = y[train_idx]
y_val = y[val_idx]

X_train_d = X_train.to(device)
y_train_d = y_train.to(device)

X_val_d = X_val.to(device)
y_val_d = y_val.to(device)
y_train_d = y_train_d.float()
y_val_d   = y_val_d.float()

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    model.train()
    optimizer.zero_grad()

    # TODO: forward
    # logits = ...
    logits = model(X_train_d)

    # TODO: loss
    # loss = ...
    loss = criterion(logits, y_train_d)

    # TODO: backward
    # loss.backward()
    loss.backward()

    # TODO: update
    # optimizer.step()
    optimizer.step()

    if epoch % 5 == 0:
        print("Epoch", epoch, "| loss =", float(loss))


Epoch 0 | loss = 0.6955855488777161
Epoch 5 | loss = 0.6945813298225403
Epoch 10 | loss = 0.6938047409057617
Epoch 15 | loss = 0.693234920501709
Epoch 20 | loss = 0.6928198337554932
Epoch 25 | loss = 0.6925092339515686
Epoch 30 | loss = 0.6922734975814819
Epoch 35 | loss = 0.6920872330665588
Epoch 40 | loss = 0.691932737827301
Epoch 45 | loss = 0.6918009519577026
Epoch 50 | loss = 0.6916845440864563
Epoch 55 | loss = 0.6915833950042725
Epoch 60 | loss = 0.6914945840835571
Epoch 65 | loss = 0.6914150714874268
Epoch 70 | loss = 0.6913353800773621
Epoch 75 | loss = 0.6912620663642883
Epoch 80 | loss = 0.6911883354187012
Epoch 85 | loss = 0.6911168694496155
Epoch 90 | loss = 0.6910359263420105
Epoch 95 | loss = 0.6909533739089966


/tmp/ipython-input-3628255691.py:42: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Epoch", epoch, "| loss =", float(loss))


## 8) Evaluation

1. Apply `sigmoid` to the logits
2. Convert probabilities to predictions
3. Compute **accuracy** on the validation set


In [ ]:
# TODO: evaluation
with torch.no_grad():
    model.eval()

    logits = model(X_val_d)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    accuracy = (preds == y_val_d).float().mean()
    accuracy

## 9) Reflection questions (answer inside the markdown)

1. Why do we **not** apply sigmoid inside the model? We do not apply sigmoid because BCEWithLogitsLoss already includes a sigmoid internally.
Keeping raw logits improves numerical stability and avoids applying sigmoid twice, which would harm learning

2. What would happen if we removed all ReLU activations? :The network would collapse into a single linear transformation, regardless of the number of layers.
It would lose all non-linearity and be unable to model complex decision boundaries.

3. How does this toy problem relate to image classification?
Each input feature here plays the role of a pixel (or learned feature) in an image.
The same pipeline (linear layers, activations, loss, backprop) scales directly to images with many pixels and deeper networks.


## 10) Bridge to Computer Vision

So far:
- inputs = vectors of size 3
- layers = fully-connected

Next session:
- inputs = images `(B, C, H, W)`
- layers = convolutions
- same training logic

👉 **Architecture changes, learning principles stay the same.**


## Part II — Training on MNIST

Check the next notebook